In [28]:
# List customer ASes and compare it with CAIDA AS realtionship
import pandas as pd
asn = "198949"
year = "2021"

confirmed_customers1 = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/confirmed_customers_as"+asn+"_"+year+".csv")
confirmed_customers1_unique_origin_ases = confirmed_customers1['origin_as'].unique()


path_prepending = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_as"+asn+"_path_prepend_01_jan_"+year+".csv")
path_prepending_unique_origin_ases = path_prepending['origin_as'].unique()
confirmed_customers2 = list(set(path_prepending_unique_origin_ases)-set(confirmed_customers1_unique_origin_ases))

df = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_not_as"+asn+"_01_jan_"+year+"_v3.csv")
sibling_path = df.loc[(df['siblings'] == 1) & (df['new_provider_sibling_check'] == int(asn))]
sibling_path_unique_origin_ases = sibling_path['origin_as'].unique()

confirmed_customers3 = list(set(sibling_path_unique_origin_ases)-set(confirmed_customers2)-set(confirmed_customers1_unique_origin_ases))

confirmed_customers = list(confirmed_customers1_unique_origin_ases) + list(confirmed_customers2) + list(confirmed_customers3)
# confirmed_customers = confirmed_customers1_unique_origin_ases # Only for Cloudlfare
# Save it into a file
with open(r"/home/shyam/jupy/ddos_scrubber/data/final_confirmed_customer_ases_"+asn+"_"+year+".txt", 'w') as fp:
    for a in confirmed_customers:
        # write each item on a new line
        fp.write("%s\n" % a)
    print('Done')

Done


In [25]:
len(confirmed_customers)

119

In [23]:
# List of ASes to check
file_path = '/home/shyam/jupy/ddos_scrubber/data/'+year+'0101.as-rel2.txt'

start_line = 181

# Read the CAIDA file starting from line 181
with open(file_path, "r") as f:
    relationships = [
        tuple(line.strip().split('|')[:3])  # Parse the first three elements
        for i, line in enumerate(f)  # Track line number
        if i >= 180 and line.strip()  # Skip lines before 100 and exclude empty lines
    ]   

# Filter for customer relationships
# customers = {int(asn2) for asn1, asn2, rel in relationships if asn1 == asn and rel == "-1"}

# Some lines do not contain 3 tuples as provider|customer|-1
customers = []
for r in relationships:
    if len(r) >= 3 and r[0] == asn and r[2] == "-1":
        customers.append(int(r[1]))

# Check which ASes in `confirmed_customers` are customers of AS32787
matching_customers = [asn for asn in confirmed_customers if asn in customers]
matching_customers = set(matching_customers)

unmatching_customers = [asn for asn in confirmed_customers if asn not in customers]
unmatching_customers = set(unmatching_customers)

# Output the results
# print(f"Matching ASes: {matching_customers}")
print(f"Number of matching ASes: {len(matching_customers)}")


# Output the results
print(f"Unmatched ASes: {unmatching_customers}")
print(f"Number of unmatched ASes: {len(unmatching_customers)}")



Number of matching ASes: 117
Unmatched ASes: {65128, 13274, 65299}
Number of unmatched ASes: 3


In [ ]:
# Check if prefix superprefix exists for those confirmed customer prefixes and has different upstream AS

import pandas as pd
import ipaddress

year = "2024"
asn = "13335"


df = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/confirmed_customers_as"+asn+"_"+year+".csv")

customer_prefixes = df[["prefix"]].copy()

customer_prefixes_unique = customer_prefixes.drop_duplicates()


def find_superprefix_provider(df, column_name):
   
    return superprefix_provider

superprefix_provider = find_superprefix_provider(customer_prefixes_unique, "prefix")



In [4]:
# Check if prefix superprefix exists for those confirmed customer prefixes

import pandas as pd
import ipaddress

year = "2024"
asn = "13335"


df = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/confirmed_customers_as"+asn+"_"+year+".csv")

customer_prefixes = df[["prefix"]].copy()

customer_prefixes_unique = customer_prefixes.drop_duplicates()
# customer_prefixes_unique
# Function to detect and return parent-child deaggregation pairs
def find_deaggregation_pairs(df, column_name):
    # Convert prefixes to ip_network objects
    prefixes = df[column_name].apply(lambda x: ipaddress.ip_network(x, strict=False)).tolist()
    deaggregation_pairs = []
    for i, parent in enumerate(prefixes):
        for j, child in enumerate(prefixes):
            # Ensure both prefixes are of the same version before comparison
            if i != j and parent.version == child.version:
                if child.subnet_of(parent) and child != parent:
                    deaggregation_pairs.append((str(parent), str(child)))
    return deaggregation_pairs

# Get deaggregation pairs
deaggregation_pairs = find_deaggregation_pairs(customer_prefixes_unique, "prefix")

print(f"Total unique customer prefixes are {len(customer_prefixes_unique)} and deaggregation pairs are {len(deaggregation_pairs)}")
# Output the results
if deaggregation_pairs:
    print("Deaggregation detected. Parent-Child pairs:")
    for parent, child in deaggregation_pairs:
        print(f"Parent: {parent}, Child: {child}")
else:
    print("No deaggregation detected.")


Total unique customer prefixes are 3321 and deaggregation pairs are 3
Deaggregation detected. Parent-Child pairs:
Parent: 103.31.6.0/23, Child: 103.31.7.0/24
Parent: 184.94.192.0/20, Child: 184.94.207.0/24
Parent: 198.41.148.0/22, Child: 198.41.148.0/24
